In [ ]:
# ================================================================
#   MULTI-MODEL PIPELINE FOR QUANTITATIVE PLANT DISEASE DATASET
#   (Works for your CSV: plant_disease_quantitative.csv)
# ================================================================

import pandas as pd
import numpy as np
import io, time, warnings, joblib
from pathlib import Path

warnings.filterwarnings("ignore")

# Colab upload helper
try:
    from google.colab import files as colab_files
except:
    colab_files = None

# sklearn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, mutual_info_classif, SelectPercentile, SelectFromModel
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, classification_report

# ---------------- USER CONFIG ----------------
CSV_PATH = None                     # set path if running locally
TARGET_COL = "Disease_Label"        # correct for your CSV
ART_DIR = Path("MLALGO/quant_model_artifacts")
ART_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_CSV = ART_DIR / "training_summary_quant_models.csv"

# ---------------- LOAD DATA ----------------
print("== LOADING QUANTITATIVE DATASET ==")

if CSV_PATH:
    df = pd.read_csv(CSV_PATH)
else:
    print("Upload your CSV now (e.g., plant_disease_quantitative.csv)")
    uploaded = colab_files.upload()
    filename = next(iter(uploaded))
    df = pd.read_csv(io.BytesIO(uploaded[filename]))

print("Data shape:", df.shape)
print(df.head())

# ---------------- PREPROCESSING ----------------

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found.")

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Label encode target
le = LabelEncoder().fit(y)
y_enc = le.transform(y)

joblib.dump(le, ART_DIR/"label_encoder.joblib")
print("Saved label encoder")

# Auto-detect numeric vs categorical
numeric_cols = [c for c in X.columns if X[c].dtype != "object"]
categorical_cols = [c for c in X.columns if X[c].dtype == "object"]

print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

# ---------------- FEATURE SET BUILDERS ----------------

feature_sets = {
    "kbest_500": SelectKBest(mutual_info_classif, k=5),        # Use 5 features only (CSV is small)
    "percentile_20": SelectPercentile(mutual_info_classif, percentile=20),
    "extratrees": SelectFromModel(ExtraTreesClassifier(n_estimators=100)),
}

# ---------------- CLASSIFIERS ----------------
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier
)

classifiers = {
    "LogReg": LogisticRegression(max_iter=2000),
    "GaussianNB": GaussianNB(),
    "SGD": SGDClassifier(),
    "DecisionTree": DecisionTreeClassifier(),
    "KNN": KNeighborsClassifier(),
    "RandomForest": RandomForestClassifier(n_estimators=100),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=100),
    "GradientBoost": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "SVC_linear": SVC(kernel="linear", probability=True),
}

# ---------------- TRAINING ----------------

summary_rows = []

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

print("\n=== TRAINING MODELS ===")

for fs_name, selector in feature_sets.items():

    print(f"\n--- FEATURE SET: {fs_name} ---")

    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ])

    # Fit preprocessing
    X_train_prep = preprocessor.fit_transform(X_train)
    X_test_prep = preprocessor.transform(X_test)

    # Apply feature selection
    selector.fit(X_train_prep, y_train)
    X_train_fs = selector.transform(X_train_prep)
    X_test_fs = selector.transform(X_test_prep)

    # Save selector + preprocessor
    joblib.dump(preprocessor, ART_DIR / f"{fs_name}_preprocessor.joblib")
    joblib.dump(selector, ART_DIR / f"{fs_name}_selector.joblib")

    print("Feature shape:", X_train_fs.shape)

    for model_name, model in classifiers.items():
        print(f" Training {model_name}...")

        start = time.time()
        model.fit(X_train_fs, y_train)
        pred = model.predict(X_test_fs)
        acc = accuracy_score(y_test, pred)
        t = time.time() - start

        print(f"  -> ACCURACY: {acc:.4f}")

        # Save model
        joblib.dump(model, ART_DIR / f"{fs_name}__{model_name}.joblib")

        summary_rows.append({
            "feature_set": fs_name,
            "model": model_name,
            "accuracy": acc,
            "time_sec": t,
            "shape": str(X_train_fs.shape),
        })

# SAVE SUMMARY
pd.DataFrame(summary_rows).to_csv(SUMMARY_CSV, index=False)
print("\nSaved summary to:", SUMMARY_CSV)
print("All artifacts saved in:", ART_DIR)


== LOADING QUANTITATIVE DATASET ==
Upload your CSV now (e.g., plant_disease_quantitative.csv)


Saving plant_disease_quantitative.csv to plant_disease_quantitative (2).csv
Data shape: (100000, 9)
   Plant_ID Plant_Species  Temperature_C  Humidity_Pct  Rainfall_mm  Soil_pH  \
0         1        Tomato           21.2          55.0         31.7     6.60   
1         2        Potato           25.7          55.1         24.9     6.90   
2         3        Tomato           30.7          45.2         12.3     7.33   
3         4        Tomato           20.8          82.7         94.6     5.71   
4         5  Corn (Maize)           13.4          71.3         45.4     5.84   

   Lesion_Size_mm  Affected_Area_Pct           Disease_Label  
0             0.0                0.0                 Healthy  
1             0.0                0.0                 Healthy  
2             0.0               67.8  Yellow Leaf Curl Virus  
3            38.0               43.2             Late Blight  
4             3.7               20.7             Common Rust  
Saved label encoder
Numeric: ['Plant_ID',

In [ ]:
import zipfile
import os
from google.colab import files as colab_files

artifact_dir = "MLALGO/quant_model_artifacts"
zip_filename = "quant_model_artifacts.zip"

# Create a ZipFile object
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, filenames in os.walk(artifact_dir):
        for filename in filenames:
            file_path = os.path.join(root, filename)
            zipf.write(file_path, os.path.relpath(file_path, artifact_dir))

print(f"Successfully created {zip_filename}")

# Offer the file for download
colab_files.download(zip_filename)

Successfully created quant_model_artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>